# 01 · Tu primer modelo de Machine Learning, de punta a punta

**Módulo 1 · Sesión 1** — Introducción al Machine Learning

## Objetivos

Al terminar este notebook habrás recorrido, en menos de 30 líneas de código útil, el
camino completo de un proyecto de Machine Learning supervisado:

1. Cargar unos datos y entender qué hay en ellos.
2. Separar lo que el modelo puede ver (**entrenamiento**) de lo que no (**prueba**).
3. Ajustar un modelo (`fit`) y usarlo para predecir (`predict`).
4. Medir qué tan bien predice, sobre datos que nunca vio.
5. Compararlo contra una referencia trivial, para saber si de verdad aprendió algo.

> **Sobre el algoritmo.** Vamos a usar una regresión lineal como **caja negra**: aquí solo
> nos interesa el *flujo*, no cómo funciona por dentro. El porqué llega en la sesión 6. Esto
> es deliberado: en Machine Learning el flujo de trabajo importa tanto como el algoritmo, y
> equivocarse en el flujo arruina hasta al mejor modelo.

## Paquetes

`pandas`, `numpy`, `matplotlib`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-1-fundamentos-ciclo-vida/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEMILLA = 42

## 1. Los datos

Trabajamos con `rendimiento-estudiantes.csv`: 400 estudiantes ficticios de posgrado, con
información de su perfil y su nota final. Son datos **simulados** con una semilla fija (ver
`datos/generar-rendimiento-estudiantes.py`), lo que nos da una ventaja pedagógica enorme:
conocemos el proceso que los generó y podremos comprobar si el modelo lo descubre.

**La pregunta:** ¿podemos predecir la nota final de un estudiante a partir de su perfil?

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
print(f"Filas: {datos.shape[0]}  ·  Columnas: {datos.shape[1]}")
datos.head()

Antes de modelar, siempre: ¿qué tipo tiene cada columna y falta algún dato?

In [ ]:
datos.info()

No hay valores faltantes (los datos reales rara vez son así de amables; el módulo 2 se
dedica justamente a eso). Veamos las variables numéricas.

In [ ]:
datos.describe().round(2)

## 2. Encuadrar el problema

Este es el paso que más se salta la gente y el que más caro se paga. Antes de escribir una
sola línea de modelado hay que responder tres preguntas:

| Pregunta | Respuesta en este caso |
|---|---|
| ¿Qué queremos predecir? | `nota_final`, un número continuo entre 0 y 5 |
| ¿Con qué información? | Perfil del estudiante: promedio previo, horas de estudio, asistencia, si trabaja |
| ¿Qué tipo de tarea es? | Aprendizaje **supervisado** de **regresión** (el objetivo es numérico) |

Si `nota_final` fuera "aprobó / no aprobó", sería una tarea de **clasificación**. El dataset
trae también esa columna (`aprobo`) para cuando lleguemos al módulo 4.

### Qué NO usamos como variable predictora

- `id_estudiante`: es un identificador. No contiene información, y si el modelo le
  encontrara alguna sería porque memorizó, no porque aprendió.
- `aprobo`: se calcula **a partir de** `nota_final`. Usarla sería hacer trampa: le estaríamos
  dando al modelo la respuesta disfrazada. Este error tiene nombre — **fuga de datos**
  (*data leakage*) — y lo estudiaremos a fondo en la sesión 5.

Empezamos con cuatro variables numéricas, las más directamente relacionadas con el estudio.

In [ ]:
caracteristicas = ["promedio_anterior", "horas_estudio_semana", "asistencia_pct", "trabaja"]
objetivo = "nota_final"

X = datos[caracteristicas]
y = datos[objetivo]

print("X (características):", X.shape)
print("y (objetivo):       ", y.shape)

> **Notación.** Por convención, `X` mayúscula es la matriz de características (una fila por
> observación, una columna por variable) y `y` minúscula el vector objetivo. La verás así en
> toda la documentación de scikit-learn y en el resto del curso.

## 3. Separar entrenamiento y prueba

Aquí está la idea central de todo el Machine Learning supervisado:

> Un modelo no se juzga por qué tan bien reproduce los datos que ya vio, sino por qué tan
> bien predice datos que **nunca** vio.

Por eso apartamos un 20 % de los estudiantes *antes* de entrenar. Ese conjunto de prueba se
guarda bajo llave y solo se toca al final, para evaluar. Es nuestra única forma honesta de
estimar cómo se comportará el modelo con estudiantes futuros.

In [ ]:
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA
)

print(f"Entrenamiento: {len(X_entrena)} estudiantes")
print(f"Prueba:        {len(X_prueba)} estudiantes")

> **Sobre `random_state=SEMILLA`.** La partición es aleatoria. Fijando la semilla,
> obtenemos siempre la misma partición y por tanto los mismos resultados. Sin esto, cada
> ejecución daría números distintos y sería imposible reproducir o comparar. Es una regla
> del curso: **toda fuente de aleatoriedad lleva semilla fija**.

## 4. Entrenar

En scikit-learn todos los modelos comparten la misma interfaz: se crean, se ajustan con
`.fit(X, y)` y predicen con `.predict(X)`. Esa uniformidad es la razón por la que cambiar de
algoritmo cuesta una línea.

In [ ]:
modelo = LinearRegression()
modelo.fit(X_entrena, y_entrena)

print("Modelo entrenado.")

Ya está. El modelo aprendió un coeficiente por cada característica.

In [ ]:
coeficientes = pd.DataFrame(
    {"caracteristica": caracteristicas, "coeficiente": modelo.coef_.round(4)}
)
print(f"Intercepto: {modelo.intercept_:.4f}\n")
print(coeficientes.to_string(index=False))

### El momento de la verdad

Como los datos son simulados, **conocemos los coeficientes reales** que los generaron.
Comparemos lo que el modelo estimó con la realidad:

In [ ]:
reales = {
    "promedio_anterior": 0.62,
    "horas_estudio_semana": 0.055,
    "asistencia_pct": 0.011,
    "trabaja": -0.22,
}
comparacion = pd.DataFrame(
    {
        "caracteristica": caracteristicas,
        "real": [reales[c] for c in caracteristicas],
        "estimado": modelo.coef_.round(4),
    }
)
comparacion["error"] = (comparacion["estimado"] - comparacion["real"]).round(4)
print(comparacion.to_string(index=False))

Los coeficientes estimados están muy cerca de los reales. **Eso es aprender**: a partir de
320 observaciones ruidosas, el algoritmo recuperó la estructura que generó los datos, sin
que nadie se la dijera.

En un problema real nunca podrás hacer esta comparación —no conoces la verdad—, y por eso
necesitas el conjunto de prueba.

## 5. Predecir y evaluar

In [ ]:
y_predicho = modelo.predict(X_prueba)

resultados = pd.DataFrame(
    {"nota_real": y_prueba.values, "nota_predicha": y_predicho.round(2)}
).head(10)
resultados["error"] = (resultados["nota_predicha"] - resultados["nota_real"]).round(2)
print(resultados.to_string(index=False))

Ahora resumimos ese error en un número. Tres métricas habituales en regresión:

- **MAE** (error absoluto medio): en promedio, ¿cuánto nos equivocamos? Está en las mismas
  unidades que la nota, así que es la más fácil de comunicar.
- **RMSE** (raíz del error cuadrático medio): parecida, pero castiga más los errores
  grandes.
- **$R^2$**: qué fracción de la variabilidad de la nota explica el modelo. 1 es perfecto,
  0 es "tan bueno como predecir siempre el promedio".

Las estudiaremos con detalle en la sesión 6.

In [ ]:
mae = mean_absolute_error(y_prueba, y_predicho)
rmse = np.sqrt(mean_squared_error(y_prueba, y_predicho))
r2 = r2_score(y_prueba, y_predicho)

print(f"MAE : {mae:.3f} puntos de nota")
print(f"RMSE: {rmse:.3f}")
print(f"R2  : {r2:.3f}")

## 6. ¿Es bueno? Compara siempre contra una referencia

El $R^2$ que acabamos de obtener suena razonable, pero **un número solo no significa nada
sin comparación**. La
referencia mínima: un modelo que ignora las características y siempre predice la nota
promedio del conjunto de entrenamiento.

Si tu modelo no le gana a esto, no ha aprendido nada.

In [ ]:
referencia = DummyRegressor(strategy="mean")
referencia.fit(X_entrena, y_entrena)
y_referencia = referencia.predict(X_prueba)

mae_ref = mean_absolute_error(y_prueba, y_referencia)
r2_ref = r2_score(y_prueba, y_referencia)

comparativa = pd.DataFrame(
    {
        "modelo": ["Referencia (predice la media)", "Regresión lineal"],
        "MAE": [round(mae_ref, 3), round(mae, 3)],
        "R2": [round(r2_ref, 3), round(r2, 3)],
    }
)
print(comparativa.to_string(index=False))
print(f"\nReducción del error: {(1 - mae / mae_ref):.1%}")

El modelo reduce el error de forma sustancial frente a la referencia. Ahora sí podemos decir
que aprendió algo útil.

> **Costumbre para todo el curso.** Empieza *siempre* por una referencia trivial
> (`DummyRegressor` o `DummyClassifier`). Es barato, y te ahorra celebrar modelos que no
> sirven. En clasificación con clases desbalanceadas esta costumbre es directamente
> obligatoria: un 95 % de acierto puede ser peor que inútil.

## 7. Mirar los errores, no solo la métrica

Dos gráficas que vale la pena hacer siempre en regresión.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

# Predicho vs. real: los puntos deberían caer sobre la diagonal.
ejes[0].scatter(y_prueba, y_predicho, alpha=0.6, edgecolor="none")
limites = [y.min() - 0.2, y.max() + 0.2]
ejes[0].plot(limites, limites, "r--", linewidth=1, label="Predicción perfecta")
ejes[0].set_xlabel("Nota real")
ejes[0].set_ylabel("Nota predicha")
ejes[0].set_title("Predicho vs. real")
ejes[0].legend()

# Residuales: deberían repartirse sin patrón alrededor de cero.
residuales = y_prueba - y_predicho
ejes[1].scatter(y_predicho, residuales, alpha=0.6, edgecolor="none")
ejes[1].axhline(0, color="r", linestyle="--", linewidth=1)
ejes[1].set_xlabel("Nota predicha")
ejes[1].set_ylabel("Residual (real - predicho)")
ejes[1].set_title("Residuales")

plt.tight_layout()
plt.show()

En la primera gráfica los puntos se agrupan alrededor de la diagonal: el modelo acierta.
En la segunda, los residuales se reparten alrededor de cero sin forma de embudo ni de curva,
que es justo lo que queremos. Cuando aparece un patrón en los residuales, el modelo se está
dejando algo sin capturar. Esto se vuelve una herramienta de diagnóstico formal en la
sesión 6.

## 8. Usar el modelo

Un modelo entrenado sirve para predecir sobre casos nuevos. Así se vería para tres
estudiantes que acaban de matricularse:

In [ ]:
nuevos = pd.DataFrame(
    {
        "promedio_anterior": [4.30, 3.10, 3.60],
        "horas_estudio_semana": [15.0, 4.0, 9.0],
        "asistencia_pct": [95.0, 62.0, 85.0],
        "trabaja": [0, 1, 1],
    }
)
nuevos["nota_predicha"] = modelo.predict(nuevos).round(2)
print(nuevos.to_string(index=False))

> **Advertencia.** Predecir es fácil; predecir con responsabilidad no. Un modelo así, usado
> para decidir sobre personas reales, exige preguntarse por el sesgo de los datos, por el
> efecto de la decisión sobre el estudiante y por la incertidumbre de cada predicción. Un
> número no es un veredicto. Volveremos sobre esto al hablar de interpretabilidad (sesión
> 11) y de monitoreo en producción (sesión 14).

## Lo que acabas de hacer

| Paso | Código | Idea |
|---|---|---|
| Encuadrar | — | ¿Qué predigo, con qué, y qué tipo de tarea es? |
| Separar | `train_test_split` | Evaluar sobre datos no vistos |
| Entrenar | `.fit(X, y)` | El modelo aprende de los datos de entrenamiento |
| Predecir | `.predict(X)` | Aplicar lo aprendido |
| Evaluar | MAE, RMSE, $R^2$ | ¿Qué tan lejos quedan las predicciones? |
| Comparar | `DummyRegressor` | ¿Le gana a lo trivial? |
| Diagnosticar | Gráfica de residuales | ¿Se le escapa algún patrón? |

Este esqueleto no cambia en todo el curso. Lo que cambia es qué modelo va en el medio, cómo
se preparan los datos y cómo se valida con más rigor.

## Para practicar

1. Añade `edad` y `estrato` a las características. ¿Mejora el $R^2$? ¿Por qué crees que sí
   o por qué no?
2. Cambia `random_state` a 7, 13 y 99. ¿Cuánto varía el $R^2$? ¿Qué te dice eso sobre
   confiar en una sola partición? (Esa inquietud es la que resuelve la validación cruzada en
   la sesión 8.)
3. Entrena solo con `promedio_anterior`. ¿Cuánto se pierde al usar una sola variable?
4. Reemplaza `LinearRegression()` por `sklearn.tree.DecisionTreeRegressor(random_state=42)`.
   Cambia una sola línea: esa es la ventaja de la interfaz uniforme. ¿Mejora?